# Training Models with Azure Machine Learning SDK v2

This notebook is the updated SDK v2 version of the older Azure ML training lab.

The old notebook used:

- `azureml.core.Workspace`
- `Experiment`
- `Estimator`
- `SKLearn`
- `Run.get_context()`
- `run.register_model()`

Those are SDK v1 patterns. This version uses:

- `azure.ai.ml.MLClient`
- `command()` jobs
- MLflow logging
- Azure ML model assets

The goal is the same: train a diabetes classification model, log metrics, save the model, and register it in Azure Machine Learning.

## 1. Install/Import Required Packages

Run this only if your environment does not already have the Azure ML SDK v2 packages.

In [ ]:
# If needed, uncomment and run this cell once.
# %pip install azure-ai-ml azure-identity mlflow pandas scikit-learn matplotlib joblib

## 2. Connect to Your Azure ML Workspace

This is the SDK v2 replacement for:

```python
ws = Workspace.from_config()
```

It expects a `config.json` file in the same folder as this notebook, or in `.azureml/config.json`.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Connect to the Azure ML workspace using config.json
ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

print("Connected to workspace:", ml_client.workspace_name)
print("Resource group:", ml_client.resource_group_name)
print("Subscription:", ml_client.subscription_id)

## 3. Choose Compute

In SDK v1, the old lab used:

```python
compute_target='local'
```

In SDK v2, you should normally run jobs on Azure ML compute. First list the available compute targets, then set `compute_name` to one that exists.

If your workspace supports serverless jobs, you can try:

```python
compute_name = "serverless"
```

In [ ]:
# List available compute targets
for compute in ml_client.compute.list():
    print(compute.name, "-", compute.type)

In [ ]:
# Change this to match your workspace compute target.
# Example: "cpu-cluster", "cpu-cluster-01", or "serverless"
compute_name = "serverless"

print("Using compute:", compute_name)

## 4. Create the Training Folder

The training job needs a folder that contains:

- the training script
- the dataset file

This assumes your notebook already has `data/diabetes.csv`, same as the original lab.

In [ ]:
import os
import shutil
from pathlib import Path

training_folder = Path("diabetes-training")
training_folder.mkdir(exist_ok=True)

data_source = Path("data/diabetes.csv")
data_target = training_folder / "diabetes.csv"

if not data_source.exists():
    raise FileNotFoundError("Missing data/diabetes.csv. Make sure the data folder is beside this notebook.")

shutil.copy(data_source, data_target)
print("Training folder ready:", training_folder)

## 5. Create the Training Script

This replaces the old SDK v1 script that used:

```python
from azureml.core import Run
run = Run.get_context()
run.log(...)
```

In SDK v2, use **MLflow**:

```python
mlflow.log_metric(...)
mlflow.sklearn.log_model(...)
```

In [ ]:
%%writefile diabetes-training/diabetes_training.py
import argparse
import os
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--reg_rate", type=float, default=0.01)
    args = parser.parse_args()

    reg = args.reg_rate

    print("Loading data...")
    diabetes = pd.read_csv("diabetes.csv")

    feature_columns = [
        "Pregnancies",
        "PlasmaGlucose",
        "DiastolicBloodPressure",
        "TricepsThickness",
        "SerumInsulin",
        "BMI",
        "DiabetesPedigree",
        "Age",
    ]

    X = diabetes[feature_columns].values
    y = diabetes["Diabetic"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=0
    )

    print(f"Training Logistic Regression model with regularization rate: {reg}")

    model = LogisticRegression(C=1 / reg, solver="liblinear")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_scores = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_scores)

    print("Accuracy:", accuracy)
    print("AUC:", auc)

    # Log parameters and metrics to Azure ML using MLflow
    mlflow.log_param("regularization_rate", reg)
    mlflow.log_metric("Accuracy", accuracy)
    mlflow.log_metric("AUC", auc)

    # Save model as a normal file artifact
    os.makedirs("outputs", exist_ok=True)
    model_path = "outputs/diabetes_model.pkl"
    joblib.dump(model, model_path)

    # Log model in MLflow format as well
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="diabetes_model"
    )

    print("Model saved to", model_path)


if __name__ == "__main__":
    main()

## 6. Create an Environment

This replaces the old SDK v1 estimator package setting:

```python
conda_packages=['scikit-learn']
```

In SDK v2, the job should use a clearly defined environment.

In [ ]:
%%writefile diabetes-training/environment.yml
name: diabetes-training-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - pandas
      - scikit-learn
      - mlflow
      - azureml-mlflow
      - joblib

## 7. Run the Training Job

This replaces the old SDK v1 block:

```python
from azureml.train.estimator import Estimator
from azureml.core import Experiment

estimator = Estimator(...)
experiment = Experiment(...)
run = experiment.submit(config=estimator)
```

In SDK v2, submit a command job.

In [ ]:
from azure.ai.ml import command
from azure.ai.ml.entities import Environment

job_env = Environment(
    name="diabetes-training-env",
    description="Environment for diabetes training with scikit-learn and MLflow",
    conda_file="diabetes-training/environment.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
)

job = command(
    code="./diabetes-training",
    command="python diabetes_training.py --reg_rate 0.01",
    environment=job_env,
    compute=compute_name,
    experiment_name="diabetes-training",
    display_name="diabetes-training-sdk-v2"
)

returned_job = ml_client.jobs.create_or_update(job)
print("Submitted job:", returned_job.name)
print("Studio URL:", returned_job.studio_url)

## 8. Stream Job Output

This is the SDK v2 replacement for:

```python
run.wait_for_completion(show_output=True)
```

In [ ]:
ml_client.jobs.stream(returned_job.name)

## 9. Get Job Details

In SDK v2, a submitted run is a **job**. You can retrieve it by name.

In [ ]:
finished_job = ml_client.jobs.get(returned_job.name)

print("Job name:", finished_job.name)
print("Status:", finished_job.status)
print("Experiment:", finished_job.experiment_name)
print("Studio URL:", finished_job.studio_url)

## 10. Run a Parameterized Training Job

The old notebook used `SKLearn` estimator with:

```python
script_params={'--reg_rate': 0.1}
```

In SDK v2, just change the command argument.

In [ ]:
job_param = command(
    code="./diabetes-training",
    command="python diabetes_training.py --reg_rate 0.1",
    environment=job_env,
    compute=compute_name,
    experiment_name="diabetes-training",
    display_name="diabetes-training-sdk-v2-reg-0-1"
)

returned_param_job = ml_client.jobs.create_or_update(job_param)
print("Submitted parameterized job:", returned_param_job.name)
print("Studio URL:", returned_param_job.studio_url)

In [ ]:
ml_client.jobs.stream(returned_param_job.name)

## 11. Register the Model

The old notebook used:

```python
run.register_model(...)
```

In SDK v2, register a model asset from the job output.

The training script saves the model file to `outputs/diabetes_model.pkl`. Azure ML automatically captures job outputs and artifacts.

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# Register the model from the job artifact path.
# This points to the model file saved by the training script.
model = Model(
    path=f"azureml://jobs/{returned_job.name}/outputs/artifacts/paths/outputs/diabetes_model.pkl",
    name="diabetes_model",
    description="Diabetes classification model trained with Azure ML SDK v2",
    type=AssetTypes.CUSTOM_MODEL,
    tags={"training_context": "SDK v2 command job"}
)

registered_model = ml_client.models.create_or_update(model)

print("Registered model:", registered_model.name)
print("Version:", registered_model.version)

## 12. List Registered Model Versions

In [ ]:
for model_version in ml_client.models.list(name="diabetes_model"):
    print("Model:", model_version.name)
    print("Version:", model_version.version)
    print("Type:", model_version.type)
    print("Tags:", model_version.tags)
    print("---")

## What Changed from the Old Version?

| Old SDK v1 | New SDK v2 |
|---|---|
| `Workspace.from_config()` | `MLClient.from_config()` |
| `Experiment` | `experiment_name` on the job |
| `Estimator` | `command()` job |
| `SKLearn` estimator | `command()` job with arguments |
| `Run.get_context()` | MLflow logging |
| `run.log()` | `mlflow.log_metric()` / `mlflow.log_param()` |
| `run.register_model()` | `ml_client.models.create_or_update()` |

The big idea is simple: **SDK v2 treats training as a job, not as an estimator-run object.**